In [2]:
print("start imports")
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm
from umap import UMAP
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import euclidean
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "browser"

start imports


C:\conda_envs\master_thesis\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
META_COLS = ["id", "genre", "popularity", "release", "artist", "song", "album_name"]

def load_features(name):
    data = pd.read_parquet(f"../data/{name}.parquet")
    features = [col for col in data.columns if col not in META_COLS]
    print(f"loaded {name}")
    return data, features

# Audio
mfcc_data, mfcc_features = load_features("mfcc")
musicnn_data, musicnn_features = load_features("musicnn")
# Video
resnet_data, resnet_features = load_features("resnet")
vgg19_data, vgg19_features = load_features("vgg19")
# Lyrics
word2vec_data, word2vec_features = load_features("word2vec")

In [7]:
def run_umap_pipeline(data, features, df, n_top_genres=12, year_bin_size=5):
    print("Scaling...")
    X = StandardScaler().fit_transform(data[features])

    print("Fitting UMAP...")
    X_2d = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X)

    df = df.copy()
    df["x"] = X_2d[:, 0]
    df["y"] = X_2d[:, 1]
    df["year_bin"] = ((df["release"] // year_bin_size) * year_bin_size).astype(int)

    top_genres = df["genre"].value_counts().nlargest(n_top_genres).index
    df_small = df[df["genre"].isin(top_genres)].sort_values("year_bin")

    centroids = (
        df_small
        .groupby(["year_bin", "genre"])
        .agg(x=("x", "mean"), y=("y", "mean"), count=("x", "size"))
        .reset_index()
    )

    fig = px.scatter(
        centroids,
        x="x", y="y",
        color="genre",
        size="count",
        hover_data=["count"],
        animation_frame="year_bin",
        category_orders={"year_bin": sorted(centroids["year_bin"].unique())},
        title="Genre Evolution (Centroids, 5-Year Steps)"
    )
    fig.update_layout(
        xaxis=dict(range=[centroids["x"].min(), centroids["x"].max()]),
        yaxis=dict(range=[centroids["y"].min(), centroids["y"].max()])
    )
    fig.show()
    print("Done.")

    return df, centroids

In [ ]:
df_mfcc, centroids_mfcc = run_umap_pipeline(mfcc_data, mfcc_features, mfcc_data)

In [ ]:
df_musicnn, centroids_musicnn = run_umap_pipeline(musicnn_data, musicnn_features, musicnn_data)

In [ ]:
df_resnet, centroids_resnet = run_umap_pipeline(resnet_data, resnet_features, resnet_data)

In [8]:
df_vgg19, centroids_vgg19 = run_umap_pipeline(vgg19_data, vgg19_features, vgg19_data)

Scaling...


MemoryError: Unable to allocate 6.07 GiB for an array with shape (98877, 8242) and data type float64

In [ ]:
df_w2v, centroids_w2v = run_umap_pipeline(word2vec_data, word2vec_features, word2vec_data)

In [13]:
def compute_centroids(df, n_top_genres=12, year_bin_size=5):
    df = df.copy()
    df = df.dropna(subset=["release", "x", "y", "genre"])
    df["year_bin"] = ((df["release"] // year_bin_size) * year_bin_size).astype(int)
    
    top_genres = df["genre"].value_counts().nlargest(n_top_genres).index
    df_small = df[df["genre"].isin(top_genres)].sort_values("year_bin")
    
    centroids = (
        df_small
        .groupby(["year_bin", "genre"])
        .agg(x=("x", "mean"), y=("y", "mean"), count=("x", "size"))
        .reset_index()
    )
    return centroids
    
centroids_vgg19 = compute_centroids(df_vgg19)
print("done")

KeyError: ['x', 'y']

In [ ]:
from itertools import combinations

def centroid_spread_over_time(centroids):
    rows = []
    for year, group in centroids.groupby("year_bin"):
        coords = group[["x", "y"]].values
        if len(coords) < 2:
            continue
        dists = [euclidean(a, b) for a, b in combinations(coords, 2)]
        rows.append({"year_bin": year, "mean_dist": np.mean(dists)})
    
    spread_df = pd.DataFrame(rows)
    fig = px.line(spread_df, x="year_bin", y="mean_dist", title="Visual Divergence Over Time")
    fig.show()
    return spread_df

spread_vgg19 = centroid_spread_over_time(centroids_vgg19)